# 1. Invalid JSON

Replay the saved parsing failure, inspect the reconstructed request, and optionally make a fresh model call. Offline cells are free. The live call is disabled until you set RUN_MODEL_CALL = True.


In [1]:
import site
from pathlib import Path

venv = Path("/home/hema/research_finder/.venv")
packages = list(venv.glob("lib/python*/site-packages"))
assert packages, "Project environment not found"
site.addsitedir(str(packages[0]))

import openai
import pymupdf
print("Dependencies loaded")

Dependencies loaded


In [2]:
from pathlib import Path
import sys
import json
from pprint import pprint

# Works when opened from the repository, notebooks/, or its parent directory.
root = next(p for base in [Path.cwd(), *Path.cwd().parents]
            for p in [base, base / "research_finder"]
            if (p / "novelty/comparison_workflow.py").exists())
sys.path.insert(0, str(root))
from notebooks.example_helpers import load_example, expect_failure
from extraction.research_extract import _parse_json
from novelty.comparison_records import EvidenceReport, EvidenceRecord, EvidenceReview
from novelty.comparison_schemas import Passage
from novelty.comparison_workflow import validate_evidence_report, validate_record, eligible, scope_citation_gaps
from novelty.schemas import TargetSignature


In [3]:
example = load_example(root, "01_invalid_json")
pprint(example["provenance"])

{'authenticity': 'actual saved failed review response; may be an intermediate '
                 'attempt',
 'checkpoint': 'comparison.json',
 'checkpoint_sha256': '9fb5f6b9ffc43d14a7e1b606e9a5cb29e6f3f237a2ef2285c56b1338bea35c67',
 'direction_id': 'dir-001',
 'paper_id': 'da66e7a58616',
 'run': 'cost_aware_agent_rl_20261011T034213Z',
 'source_pointer': '/result/candidates/0/papers/68/evidence_reviews/1/format_repairs/0'}


## Inspect the saved output
The complete original response is in `example['invalid_output']`. We display its ending because the response is long.

In [4]:
raw = example["invalid_output"]
print("Recorded error:", example["recorded_error"])
print("Response ending:", repr(raw[-250:]))

Recorded error: Extra data: line 1 column 33817 (char 33816)
Response ending: ' for the candidate\'s cost/task variance-ratio intervention. The target result therefore remains honestly not_assessed rather than supported or refuted, and this assessment does not assert that variance-ratio effects are absent from the literature."}}'


## Reproduce the production parser failure

In [5]:
error = expect_failure(lambda: _parse_json(raw), "Extra data")
first_object, end = json.JSONDecoder().raw_decode(raw.strip())
print("Trailing text after first JSON object:", repr(raw.strip()[end:]))

Expected JSONDecodeError:
Extra data: line 1 column 33817 (char 33816)
Trailing text after first JSON object: '}'


The parser never reaches schema or scientific checks. Decoding the first object above is a diagnostic only: it does not establish that the object meets the schema or contains valid evidence. This saved attempt could have been followed by a repair; it is not a count of unique failed papers.

## Reproduce this specific model request
[Open the full input and prompt](examples/01_model_request.json).

This is the **evidence-review** call that produced the malformed response. The original wire request was not logged. Input is reconstructed from the exact saved record and targets, hash-verified source pages, and current production prompt/schema. Historical environment-dependent temperature and output caps were not saved, so this is not a guaranteed byte-for-byte historical request.

Inspect or edit the prompt/input below, run the model explicitly, then validate its new response. A fresh response may succeed or fail differently.

In [6]:
request = load_example(root, "01_model_request")
system_prompt = request["system_prompt"]
user_input = request["user_input"]
pprint(request["provenance"])
print("Provider:", request["provider"], "Model:", request["model"])
print("Task:", user_input["task"])
print("Targets:", [t["target_id"] for t in user_input["payload"]["targets"]])
print("Claims:", len(user_input["record"]["claims"]))
print("Prior passages:", len(user_input["payload"]["prior_passages"]))
print("Candidate-source passages:", len(user_input["payload"]["candidate_source_passages"]))

{'authenticity': 'actual saved failed review response; may be an intermediate '
                 'attempt',
 'checkpoint': 'comparison.json',
 'checkpoint_sha256': '9fb5f6b9ffc43d14a7e1b606e9a5cb29e6f3f237a2ef2285c56b1338bea35c67',
 'direction_id': 'dir-001',
 'paper_id': 'da66e7a58616',
 'request_authenticity': 'Reconstructed with current production prompt/schema '
                         'and saved record/targets; source artifact and page '
                         'hashes verified. Original wire request and '
                         'environment-dependent sampling/output caps were not '
                         'logged.',
 'run': 'cost_aware_agent_rl_20261011T034213Z',
 'source_pointer': '/result/candidates/0/papers/68/evidence_reviews/1/format_repairs/0'}
Provider: deepseek Model: deepseek-flash
Task: review_comparison_evidence
Targets: ['dir-001-h03']
Claims: 16
Prior passages: 134
Candidate-source passages: 35


## Full system prompt
Edit system_prompt to test a prompt change.

In [7]:
print(system_prompt)

Return exactly the supplied EvidenceReport schema object: claim_checks,
coverage_checks and summary. Do not add schema metadata, alternate reason fields, or
extra keys, including during format repair. Copy passage IDs exactly from supplied
passages; do not reconstruct hashes from memory.

Independently review the evidence record BEFORE overlap classification.
Return one claim_check for EVERY claim and one coverage_check for EVERY target, exactly
once each. Judge the actual cited text, not remembered paper summaries or previous reviews.
For supported claims cite inspected passage IDs already attached to that claim. Support
elsewhere is a citation repair, not supported status. Inspect every clause, qualification,
experimental condition, numerical assertion and inference assumption. Do not demand a
verbatim lexical match: scientific paraphrase is allowed when meaning is preserved.

For EVERY claim return scope_checks and scope_summary. For results, evaluation
claims and theoretical result

## Full user input
Includes the response schema, target, paper passages and claims under review. This output is long; the same input is in the linked JSON.

In [17]:
user_input.keys()

dict_keys(['task', 'schema', 'payload', 'record', 'required_scope_claim_ids'])

In [ ]:
print(json.dumps(user_input, indent=2, ensure_ascii=False))

{
  "task": "review_comparison_evidence",
  "schema": {
    "$defs": {
      "ClaimCheck": {
        "additionalProperties": false,
        "properties": {
          "claim_id": {
            "minLength": 1,
            "title": "Claim Id",
            "type": "string"
          },
          "decision": {
            "enum": [
              "supported",
              "revise",
              "unresolved"
            ],
            "title": "Decision",
            "type": "string"
          },
          "explanation": {
            "minLength": 1,
            "title": "Explanation",
            "type": "string"
          },
          "scope_checks": {
            "anyOf": [
              {
                "items": {
                  "$ref": "#/$defs/ScopeCheck"
                },
                "type": "array"
              },
              {
                "type": "null"
              }
            ],
            "default": null,
            "title": "Scope Checks"
          },
     

## Explicit live model call
Set RUN_MODEL_CALL = True and execute this cell when ready. It uses the repository DeepSeek configuration/API key and JSON response mode. Run All is free with the default False.

There is no pipeline or evidence-repair loop. SDK retries are disabled; the shared client can still retry once with a smaller output allowance after a provider context rejection. The saved requested output allowance is adjusted by the same client/provider limits as the pipeline. Current environment settings apply. No API keys are printed or saved.

New requests and responses are saved under notebooks/local_results/, separately from the historical fixture.

In [23]:
request["user_input"]['schema']

{'$defs': {'ClaimCheck': {'additionalProperties': False,
   'properties': {'claim_id': {'minLength': 1,
     'title': 'Claim Id',
     'type': 'string'},
    'decision': {'enum': ['supported', 'revise', 'unresolved'],
     'title': 'Decision',
     'type': 'string'},
    'explanation': {'minLength': 1, 'title': 'Explanation', 'type': 'string'},
    'scope_checks': {'anyOf': [{'items': {'$ref': '#/$defs/ScopeCheck'},
       'type': 'array'},
      {'type': 'null'}],
     'default': None,
     'title': 'Scope Checks'},
    'scope_summary': {'anyOf': [{'minLength': 1, 'type': 'string'},
      {'type': 'null'}],
     'default': None,
     'description': 'Explain which result conditions matter and any ambiguity. Empty checks are allowed only for non-result claims with no applicable scoped assertions.',
     'title': 'Scope Summary'},
    'passage_ids': {'description': 'Passages inspected to support this judgment; do not invent quotations.',
     'items': {'minLength': 1, 'type': 'string'},


In [24]:
user_input_new = user_input.copy()
del user_input_new["schema"]

In [28]:
user_input_new.keys()

dict_keys(['task', 'payload', 'record', 'required_scope_claim_ids'])

In [27]:
# user_input["schema"]

In [37]:
RUN_MODEL_CALL = True
MODEL = request["model"]
MAX_OUTPUT_TOKENS = request["max_tokens"]
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": json.dumps(user_input, ensure_ascii=False, separators=(",", ":"))},
]
fresh_result = None
if RUN_MODEL_CALL:
    from llm_client import AsyncLLMClient
    from datetime import datetime, timezone
    client = AsyncLLMClient(request["provider"], concurrency=1, max_retries=0)
    try:
        # fresh_result = await client.chat_result(
        #     messages=messages, model=MODEL, max_tokens=MAX_OUTPUT_TOKENS,
        #     response_format=request["response_format"],
        # )
        
        fresh_result = await client.respond(
            model=MODEL, max_output_tokens=MAX_OUTPUT_TOKENS, instructions=system_prompt,
            input = json.dumps(user_input_new),
            text = {
                "format":{
                    "type": "json_schema",
                    "name": "expected_output",
                    "schema": user_input["schema"]
                }
            }
        )


    finally:
        await client._client.close()
    output_dir = root / "notebooks" / "local_results"
    output_dir.mkdir(exist_ok=True)
    output_path = output_dir / ("01_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".json")
    output_path.write_text(json.dumps({"messages": messages, "requested_model": MODEL,
        "requested_max_tokens": MAX_OUTPUT_TOKENS, "response_format": request["response_format"],
        "text": fresh_result.text, "model": fresh_result.model,
        "finish_reason": fresh_result.finish_reason}, indent=2, ensure_ascii=False))
    print("Saved request and response:", output_path)
    print("Finish reason:", fresh_result.finish_reason)
else:
    print("Call skipped. Set RUN_MODEL_CALL = True to run it.")

AttributeError: 'str' object has no attribute 'text'

In [39]:
json.loads(fresh_result)

{'claim_checks': [{'claim_id': 'c1',
   'decision': 'supported',
   'explanation': 'The cited abstract and introduction state that GRPO in Search-R1-style tool-integrated RL consistently suffers training collapse, frequently with abrupt reward drops, and explicitly identify LLD as a systematic reduction or stagnation in the likelihood of correct response trajectories and as the core mechanism driving this failure.',
   'passage_ids': ['da66e7a58616#p1:9a1bfaa4c7b3',
    'da66e7a58616#p1:759ed5dea957'],
   'scope_checks': [{'aspect': 'outcome',
     'assertion': 'GRPO training in Search-R1-style tool-integrated RL pipelines frequently suffers abrupt reward drops and catastrophic collapse.',
     'decision': 'supported',
     'passage_ids': ['da66e7a58616#p1:759ed5dea957'],
     'explanation': 'The passage states that GRPO training in Search-R1-style pipelines frequently suffers from abrupt reward drops and catastrophic collapse.'},
    {'aspect': 'model',
     'assertion': 'The failure 

## Validate the fresh response
Identify the first failure without triggering another paid call. Passing these checks does not establish scientific correctness or final approval.

In [38]:
if fresh_result is not None:
    step = "output completeness"
    try:
        # if fresh_result.truncated:
        #     raise ValueError("truncated model response")
        step = "JSON parsing"
        parsed_fresh = _parse_json(fresh_result)
        print("JSON parsing: PASS")
        step = "schema validation"
        report_fresh = EvidenceReport.model_validate(parsed_fresh)
        print("Schema validation: PASS")
        step = "evidence-reference validation"
        saved_record = EvidenceRecord.model_validate(user_input["record"])
        validate_evidence_report(report_fresh, saved_record,
            [Passage.model_validate(p) for p in user_input["payload"]["prior_passages"]], require_scope=True)
        print("Evidence-reference validation: PASS")
        print("Scope citation gaps:", scope_citation_gaps(report_fresh, saved_record))
    except ValueError as error:
        print(step + ": FAIL")
        print(error)
else:
    print("No fresh response yet. Run the model-call cell when ready.")

JSON parsing: PASS
Schema validation: PASS
Evidence-reference validation: PASS
Scope citation gaps: {}
